In [ ]:
import numpy as np
#data loader
import pandas as pd
#will need for plotting
import matplotlib.pyplot as plt
#sklearn import for various api calls
import sklearn as sk
#we'll manually do the train_test_split
from sklearn.model_selection import train_test_split
#will grab the dataset from keras.datasets
from keras.datasets import california_housing
#different models we'll try and use:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

Getting dataset from keras california housing:
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The dataset Contains 8 input features:
<li>MedInc: median income in block group</li>
<li>HouseAge: median house age in block group</li>
<li>AveRooms: average number of rooms per household</li>
<li>AveBedrms: average number of bedrooms per household</li>
<li>Population: block group population</li>
<li>AveOccup: average number of household members</li>
<li>Latitude: block group latitude</li>
<li>Longitude: block group longitude</li>

I am picking a train_test split of 0, so that everything gets added to the training set, from there, I will be splitting via sklearn function, what we have learned in class.

<i>Ps., I could also do this autmatically by passing a seed argument to the load_data function below. </i>

In [ ]:
(input_features, output), (_, _) = california_housing.load_data(
    version="large", path="california_housing.npz", test_split=0.0
)

Since we're using the "large" version of the dataset, our data should have 20,640 entries.


In [ ]:
print("Datapoints: " , len(input_features))

Datapoints:  20640


In [ ]:
#Lets cast our features and output from tuples into np arrays so that we can call train_test_split
input_features  = np.array(input_features)
output          = np.array(output)
#Next line should tell us that our array has 20640 datapoints(rows) and 8 input features(columns)
input_features.shape

(20640, 8)

In [ ]:
#I will use this as a seed to do the split, I'll also print out the generated number for reproducability.
import random
seed = random.randint(1, 200)
print("Random seed number:", seed)
#Lets do the split:
x_train, x_test, y_train, y_test = train_test_split(
    input_features, output, test_size=0.2, random_state=seed
)
#Lets make sure that our split worked:
print("\nx_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Random seed number: 98

x_train shape: (16512, 8)
x_test shape: (4128, 8)
y_train shape: (16512,)
y_test shape: (4128,)


Let us think about our regression problem:

There is a lot of algorithms we could use to solve our regression; some examples:
*   LinearRegression (OLS)
*   Lasso(l1)
*   Ridge (l2)

Each have ups and downs, while OLS is a short "quick" fix, it has its downsides, mainly due to the fact that no normalization is required. However, it is really quick to give you a decent fit for smaller datasets like our own due to its closed form solution.

Ridge also has a closed form solution unless you pass a parameter that forces it to use GD.

Lasso is by definition an iterative method, hence it has no closed form solution.

In [ ]:
#OLS Model:
osl_model = LinearRegression()
osl_model.fit(x_train, y_train)
osl_pred = osl_model.predict(x_test)

In [ ]:
#l1 and l2 need normalization as preprocessing, we'll use the standard scaler, and create simple pipelines
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
#Lasso Model
l1_model = Lasso(alpha=0.1) #default alpha
l1_model.fit(x_train, y_train)
l1_pred = l1_model.predict(x_test)

In [ ]:
#Ridge Model
l2_model = Ridge(alpha=1.0) #default alpha
l2_model.fit(x_train, y_train)
l2_pred = l2_model.predict(x_test)

In [ ]:
#Let us compare the three:
print("---OSL Model---")
print("Mean Squared Error:", mean_squared_error(y_test, osl_pred))
print("R² Score:", r2_score(y_test, osl_pred))
print("\n---Lasso Model---")
print("Mean Squared Error:", mean_squared_error(y_test, l1_pred))
print("R² Score:", r2_score(y_test, l1_pred))
print("\n---Ridge Model---")
print("Mean Squared Error:", mean_squared_error(y_test, l2_pred))
print("R² Score:", r2_score(y_test, l2_pred))

---OSL Model---
Mean Squared Error: 11591328768.0
R² Score: 0.16944122314453125

---Lasso Model---
Mean Squared Error: 4902816256.0
R² Score: 0.6486963033676147

---Ridge Model---
Mean Squared Error: 4902977536.0
R² Score: 0.6486846804618835


In [13]:
#let us increase tune alpha, on plain ridge and lasso regressions
ridge_best,ridge_best_alpha,ridge_mse,r2_l1 = None, None, None, None
lasso_best,lasso_best_alpha,lasso_mse,r2_l2 = None, None, None, None
l1_mse_best, l2_mse_best = 999999999999, 99999999999999999
alpha = 0.1
while (alpha <= 20.1):
    l1_model = make_pipeline(StandardScaler(), Lasso(alpha))
    l1_model.fit(x_train, y_train)
    l1_pred = l1_model.predict(x_test)
    lasso_mse = mean_squared_error(y_test, l1_pred)
    if (lasso_mse < l1_mse_best ):
        r2_l1 = r2_score(y_test, l1_pred)
        lasso_best_alpha = alpha
        lasso_best = l1_model
        l1_mse_best = lasso_mse

    l2_model = make_pipeline(StandardScaler(), Ridge(alpha)) #default alpha
    l2_model.fit(x_train, y_train)
    l2_pred = l2_model.predict(x_test)
    ridge_mse = mean_squared_error(y_test, l2_pred)
    if (ridge_mse < l2_mse_best):
        r2_l2 = r2_score(y_test, l2_pred)
        ridge_best_alpha = alpha
        ridge_best = l2_model
        l2_mse_best = ridge_mse
    alpha += 0.05

#Let us print those best parameters:
print("Best alpha for Lasso:", lasso_best_alpha)
print("Lasso MSE:", l1_mse_best)
print("Lasso R^2:", r2_l1)
print("\nBest alpha for Ridge:", ridge_best_alpha)
print("Ridge MSE:", l2_mse_best)
print("Ridge R^2:", r2_l2)

Best alpha for Lasso: 0.1
Lasso MSE: 4670492672.0
Lasso R^2: 0.6284202337265015

Best alpha for Ridge: 7.149999999999983
Ridge MSE: 4670411776.0
Ridge R^2: 0.6284266710281372


In [ ]:
# Since Ridge model above reached the lowest error, we use that as our best model
y_pred = ridge_best.predict(x_test)

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual values")
plt.ylabel("Predicted values")
plt.title("Actual vs Predicted")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

From the plot above, we can see that the original data is clipped at the max of 500,001

In [ ]:
y_test.max()

So lets do a bit of clipping;


In [ ]:
y_pred = np.clip(y_pred, None, np.float32(500001.0))
print(mean_squared_error(y_test, y_pred))
print(r2_score(y_test, y_pred))

In [ ]:
#Lets make sure the clipping worked:
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual values")
plt.ylabel("Predicted values")
plt.title("Actual vs Predicted")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

In [ ]:
feature_idx = 0

sorted_idx = x_test[:, feature_idx].argsort()
x_sorted = x_test[sorted_idx, feature_idx]
y_pred_sorted = y_pred[sorted_idx]

plt.scatter(x_test[:, feature_idx], y_test, alpha=0.5, label='Actual')
plt.plot(x_sorted, y_pred_sorted, color='red', label='Fitted line')
plt.xlabel(f"Feature {feature_idx}")
plt.ylabel("Target")
plt.title(f"Regression fit for feature {feature_idx}")
plt.legend()
plt.show()